### Agents and Tool UseAn agent is an LLM whose output can trigger an ACTION (a tool call), whose result gets fed back into the model's context, run in a loop until the model decides it's done, as opposed to a single prompt-in, text-out call (`prompt-engineering.ipynb`, and `fraud-theme-detection`'s own structured-extraction calls, which are single-shot, not agentic, see the contrast at the end). Covers function calling, the agent loop, the ReAct pattern, and failure modes.

#### 0. Function calling: the schema mechanismSame JSON-schema mechanism as `prompt-engineering.ipynb`'s structured-output section, `FEATURE_SCHEMA` from `fraud-theme-detection`, and `additionalProperties: false`, just pointed at a different target. Structured-output prompting constrains the shape of a FINAL ANSWER. Function calling constrains the shape of a TOOL INVOCATION, a schema describing each available tool's name, description, and parameters, and the model picks one (or none) and fills in the arguments, instead of filling in an answer field directly.Worked schema, a toy fraud-investigation tool:```json{  "name": "get_transaction_history",  "description": "Look up recent transactions for an account",  "parameters": {    "type": "object",    "properties": {      "account_id": {"type": "string"},      "days": {"type": "integer", "description": "How many days back to look"}    },    "required": ["account_id", "days"],    "additionalProperties": false  }}```Given this schema plus a user query ("has account ACC-4471 had unusual activity lately?"), the model's response is not text, it's a structured call: `{"name": "get_transaction_history", "arguments": {"account_id": "ACC-4471", "days": 30}}`, parsed and executed by YOUR orchestration code, the model itself never touches a database, it only ever emits the intent to call one.

#### 1. The agent loop, worked by handA toy trace, 2 iterations, fraud-investigation query "has account ACC-4471 had unusual activity lately, and if so, what typology does it match?":```Iteration 1:  context so far: [user query]  model output:   tool_call get_transaction_history(account_id="ACC-4471", days=30)  orchestration:  executes the call against a real database  tool result:    [{amount: 9800, note: "wire to new payee"}, {amount: 50, note: "grocery"}, ...]  context now:    [user query, tool_call, tool_result]   <- appended, not replacing anythingIteration 2:  model output:   tool_call classify_typology(narrative="wire to new payee, $9800, ...")  orchestration:  executes the call (e.g. the fraud-theme-detection classifier itself)  tool result:    {"typology": "business_email_compromise", "confidence": 0.87}  context now:    [user query, tool_call_1, result_1, tool_call_2, result_2]Iteration 3:  model output:   final answer, no tool_call this time, "Yes, a $9800 wire to a new payee looks                  like business email compromise (87% confidence), recommend review."  loop ends:      model emitted a final answer instead of another tool_call```Each iteration is a FULL separate LLM call over the whole accumulated context (not one continuous generation), the loop's termination condition is simply "did the model's output contain a tool_call, or a final answer", checked by the orchestration code after each call.

In [ ]:
# toy orchestration loop, illustrative structure (no real API call, shows the control flow only)def get_transaction_history(account_id, days):    return [{"amount": 9800, "note": "wire to new payee"}, {"amount": 50, "note": "grocery"}]def classify_typology(narrative):    return {"typology": "business_email_compromise", "confidence": 0.87}TOOLS = {"get_transaction_history": get_transaction_history, "classify_typology": classify_typology}def run_agent_loop(user_query, mock_model_outputs, max_iterations=5):    context = [{"role": "user", "content": user_query}]    for i in range(max_iterations):        model_output = mock_model_outputs[i]  # in reality: call the LLM with `context`        if model_output["type"] == "final_answer":            print(f"[iter {i+1}] FINAL ANSWER: {model_output['content']}")            return model_output["content"]        tool_name, args = model_output["name"], model_output["arguments"]        result = TOOLS[tool_name](**args)        print(f"[iter {i+1}] tool_call: {tool_name}({args}) -> {result}")        context.append({"role": "assistant", "tool_call": model_output})        context.append({"role": "tool", "content": result})    raise RuntimeError("hit max_iterations without a final answer")mock_outputs = [    {"type": "tool_call", "name": "get_transaction_history", "arguments": {"account_id": "ACC-4471", "days": 30}},    {"type": "tool_call", "name": "classify_typology", "arguments": {"narrative": "wire to new payee, $9800"}},    {"type": "final_answer", "content": "Looks like business email compromise (87% confidence), recommend review."},]run_agent_loop("has account ACC-4471 had unusual activity lately?", mock_outputs)

#### 2. ReAct: interleaving explicit reasoning with actionsThe common academic framing for the loop above: at each step the model emits a `Thought` (reasoning about what to do next, same mechanism as `prompt-engineering.ipynb`'s chain-of-thought, now interleaved with actions instead of running once before a final answer), an `Action` (the tool call), gets back an `Observation` (the tool result), and repeats.```Thought: I need to check ACC-4471's recent transactions before I can assess risk.Action: get_transaction_history(account_id="ACC-4471", days=30)Observation: [{amount: 9800, note: "wire to new payee"}, ...]Thought: A $9800 wire to a new payee is a classic BEC pattern, worth classifying explicitly.Action: classify_typology(narrative="wire to new payee, $9800")Observation: {"typology": "business_email_compromise", "confidence": 0.87}Thought: I have enough to answer now.Final Answer: Looks like business email compromise (87% confidence), recommend review.```Making the reasoning explicit (vs. a bare tool_call with no stated justification) makes the trace auditable, useful for exactly the reason `model-interpretability.ipynb`'s SHAP section matters, being able to say WHY an automated system did what it did, here at the level of a whole multi-step decision, not just one prediction's feature attribution.

#### 3. Real function-calling API shapeOpenAI-compatible APIs (the same client pattern `fraud-theme-detection` uses for structured extraction) accept a `tools` parameter alongside the message list, and return either normal text or a `tool_calls` field on the response, the API-level version of the toy loop above.

In [ ]:
# illustrative, not executed here (needs a real API key)# from openai import OpenAI# client = OpenAI()## tools = [{#     "type": "function",#     "function": {#         "name": "get_transaction_history",#         "description": "Look up recent transactions for an account",#         "parameters": {#             "type": "object",#             "properties": {#                 "account_id": {"type": "string"},#                 "days": {"type": "integer"}#             },#             "required": ["account_id", "days"],#             "additionalProperties": False#         }#     }# }]## response = client.chat.completions.create(#     model="gpt-4o-mini",#     messages=[{"role": "user", "content": "has account ACC-4471 had unusual activity lately?"}],#     tools=tools# )## if response.choices[0].message.tool_calls:#     call = response.choices[0].message.tool_calls[0]#     print(call.function.name, call.function.arguments)  # model's chosen tool + JSON args, parse and execute# else:#     print(response.choices[0].message.content)  # model answered directly, no tool neededprint("see commented block above — the model's tool_calls field is the API-level analog")print("of the toy mock_outputs list in the agent-loop demo above")

#### 4. Model Context Protocol (MCP), brieflyA standardized protocol for exposing tools/resources to any MCP-compatible client, instead of wiring up a custom tool schema per model provider. Decouples "what tools exist" from "which model is calling them", a tool server implemented once can be used by any MCP client. This very environment (Claude Code) is itself an MCP-capable client, tool servers can be attached to it the same way a tool schema gets attached to an API call above, just standardized across providers instead of bespoke per API.

#### 5. Multi-agent patterns, brieflyInstead of one model doing everything in one loop, an orchestrator model delegates sub-tasks to separate sub-agents (each potentially with its own tools, context, even its own model), then combines their results. Useful when a task genuinely decomposes into independent pieces that don't need to share context (parallelizable), or when different sub-tasks benefit from different tool access/permissions. Added coordination complexity, a sub-agent's mistake still has to surface back to the orchestrator correctly, so this is worth reaching for only once a single agent loop genuinely can't handle the task's structure, not by default.

#### 6. Failure modes and tradeoffs- Infinite/runaway loops: no natural termination if the model keeps calling tools without ever emitting a final answer, always cap `max_iterations` (the toy loop above does this explicitly).- Tool-call hallucination: the model invents a tool name or argument that doesn't exist in the schema, orchestration code needs to validate the call against the schema before executing it, never trust it blindly.- Error handling: a tool call can fail (bad account_id, a downstream API timeout), the error needs to be fed BACK into the context as an observation, same as a successful result, so the model can adjust (retry with different arguments, or give up and say so) rather than the loop just crashing.- Cost and latency: every loop iteration is a full additional LLM call over accumulated context, an N-step agent task costs roughly N times a single structured-extraction call, and each step adds latency, not something to reach for when a single call would do.- Security: tool arguments come from model output, not directly from a trusted user, treat them like any other untrusted input, sanitize/validate before using them in a database query, shell command, or file path, exactly the injection risk any external input carries.

#### 7. When this is NOT what you want: contrast with structured extraction`fraud-theme-detection`'s own `FEATURE_SCHEMA` extraction is explicitly NOT an agent: one call, one fixed schema, no loop, no tool execution, no multi-step decision-making, it maps narrative text straight to structured features and stops. Reach for the agent loop only when the task genuinely needs actions interleaved with reasoning (look something up, then decide what to do with what you found, possibly look up something else). A single structured-output call is simpler, cheaper, faster, and easier to eval (`llm-evals.ipynb`) whenever the task doesn't actually require that back-and-forth, most classification/extraction tasks don't.